# Question 3: How close is a country to an ideal death ratio?

This notebook builds a death ratio from the healthiest nations, then measure how far every other country is from it, and see if that distance tracks development.

## Imports

In [1]:
# Core packages
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC

# Model selection and evaluation
from sklearn.model_selection import cross_val_score, GridSearchCV, LeaveOneOut, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                            roc_auc_score, confusion_matrix, classification_report,
                            mean_squared_error, r2_score, mean_absolute_error)
# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import cross_val_predict

# Other
from sklearn.metrics.pairwise import euclidean_distances
from scipy.stats import spearmanr

In [ ]:
# merged_df is the fully cleaned working dataframe. It is made in the first question file.
merged_df = pd.read_csv('merged-df.csv')

## Step 0: Data preparation

Although the dataset has already been modified, this question requires additional preparation since we need to work with probabilities rather than raw numbers.

First, we limit the scope of the data to 2019, the most recent year available prior to COVID-19, which would otherwise skew the results. We then convert the raw death counts into probabilities.

In [3]:
merged_df = merged_df[merged_df['Year']==2019]

In [4]:
disease_cols = [
    "Alzheimer's Disease and Other Dementias",
    "Parkinson's Disease",
    "Cardiovascular Diseases",
    "Neoplasms",
    "Diabetes Mellitus",
    "Chronic Kidney Disease",
    "Chronic Respiratory Diseases",
    "Cirrhosis and Other Chronic Liver Diseases",
    "HIV/AIDS"
]

merged_df[disease_cols] = merged_df[disease_cols].div(merged_df['Total Chronic Deaths'], axis=0)

merged_df

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
29,29,Afghanistan,AFG,2019,0.016554,0.005223,0.578161,0.198148,0.044923,0.052570,0.066046,0.035411,0.002966,107228,2097.889450,0.488,63.5645
59,59,Albania,ALB,2019,0.044889,0.012140,0.631682,0.230321,0.008567,0.016105,0.039896,0.016301,0.000098,20428,13485.311240,0.810,79.2825
89,89,Algeria,DZA,2019,0.033912,0.008353,0.637551,0.155047,0.034686,0.053390,0.049009,0.026334,0.001719,153605,11241.864720,0.748,76.4742
119,119,Andorra,AND,2019,0.069098,0.013436,0.324376,0.441459,0.017274,0.030710,0.074856,0.023033,0.005758,521,54465.047400,0.873,83.0039
149,149,Angola,AGO,2019,0.015695,0.003666,0.353235,0.175643,0.055380,0.033835,0.054021,0.077804,0.230721,72824,6082.746624,0.595,62.4484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5399,5399,Vanuatu,VUT,2019,0.012156,0.008317,0.555982,0.143954,0.081254,0.044786,0.110045,0.031350,0.012156,1563,3417.730001,0.611,69.8769
5429,5429,Venezuela,VEN,2019,0.045576,0.008078,0.444061,0.251023,0.077933,0.075117,0.050983,0.033873,0.013356,138517,7045.248000,0.721,72.1614
5459,5459,Yemen,YEM,2019,0.020622,0.004920,0.664197,0.151074,0.021623,0.031880,0.068359,0.033191,0.004134,83939,1349.567046,0.461,65.0917
5489,5489,Zambia,ZMB,2019,0.012123,0.003225,0.288495,0.155536,0.040882,0.030681,0.037943,0.072978,0.358136,62937,3365.410652,0.575,62.7926


## Step 1: Create the independent death ratio

We create the independent death ratio by averaging the death ratios of the 5 countries with the highest life expectancy in 2019.

In [5]:
merged_df.sort_values(by='LifeExpectancy', ascending=False).head()

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
3239,3239,Monaco,MCO,2019,0.060738,0.015184,0.349241,0.488069,0.008677,0.017354,0.043384,0.015184,0.002169,461,NaN,NaN,86.5424
2459,2459,Japan,JPN,2019,0.145534,0.014785,0.328790,0.390570,0.007754,0.040007,0.049249,0.023169,0.000142,1132892,43375.78940,0.924,84.4258
4859,4859,Switzerland,CHE,2019,0.088419,0.016641,0.412476,0.348443,0.023989,0.038169,0.052900,0.018327,0.000637,58110,68432.68312,0.962,83.7802
4469,4469,Singapore,SGP,2019,0.084395,0.013542,0.389121,0.400034,0.010228,0.043312,0.042855,0.014456,0.002057,17501,86601.44993,0.943,83.7584
4649,4649,South Korea,KOR,2019,0.073604,0.014880,0.309085,0.415575,0.051309,0.031830,0.061856,0.041312,0.000551,250679,43163.03806,0.923,83.6557


We exclude Monaco due to its relatively small population compared to most other nations, which increases the variability of its ratio.

In [6]:
ratio_df = merged_df.sort_values(by='LifeExpectancy', ascending=False).iloc[1:6].copy()

ratio_df

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
2459,2459,Japan,JPN,2019,0.145534,0.014785,0.328790,0.390570,0.007754,0.040007,0.049249,0.023169,0.000142,1132892,43375.78940,0.924,84.4258
4859,4859,Switzerland,CHE,2019,0.088419,0.016641,0.412476,0.348443,0.023989,0.038169,0.052900,0.018327,0.000637,58110,68432.68312,0.962,83.7802
4469,4469,Singapore,SGP,2019,0.084395,0.013542,0.389121,0.400034,0.010228,0.043312,0.042855,0.014456,0.002057,17501,86601.44993,0.943,83.7584
4649,4649,South Korea,KOR,2019,0.073604,0.014880,0.309085,0.415575,0.051309,0.031830,0.061856,0.041312,0.000551,250679,43163.03806,0.923,83.6557
2399,2399,Italy,ITA,2019,0.085869,0.014151,0.416476,0.338193,0.038019,0.029078,0.055197,0.021924,0.001094,567877,43121.80118,0.897,83.5520


We now compute an averaged ratio across these 5 nations.

In [7]:
ratio_df[disease_cols] = ratio_df[disease_cols].sum(axis=0)

ratio_df = ratio_df.head(1)

ratio_df = ratio_df[disease_cols]

ratio_df = ratio_df/5

/tmp/ipykernel_33498/2068937183.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ratio_df[disease_cols] = ratio_df[disease_cols].sum(axis=0)


In [8]:
ratio_df

,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS
2459,0.095564,0.0148,0.371189,0.378563,0.02626,0.036479,0.052411,0.023838,0.000896


The resulting independent death ratio is shown below.

## Step 2: Euclidean distance of all nation's ratios to the independent death ratio

We add a new column, Distance, to merged_df to evaluate how each nation compares to the independent death ratio.

In [9]:
merged_df['Distance'] = euclidean_distances(merged_df[disease_cols], ratio_df)

In [10]:
merged_df.sort_values(by='Distance')

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,Distance
269,269,Australia,AUS,2019,0.079215,0.016935,0.363760,0.378454,0.029875,0.036697,0.078954,0.015610,0.000500,141893,47312.192000,0.941,83.1100,0.033355
2849,2849,Luxembourg,LUX,2019,0.070365,0.016642,0.388905,0.378102,0.019854,0.034745,0.064526,0.026277,0.000584,3425,76018.564370,0.927,82.1434,0.033901
4469,4469,Singapore,SGP,2019,0.084395,0.013542,0.389121,0.400034,0.010228,0.043312,0.042855,0.014456,0.002057,17501,86601.449930,0.943,83.7584,0.037326
509,509,Belgium,BEL,2019,0.073728,0.014835,0.374791,0.377745,0.019293,0.033047,0.082477,0.023461,0.000624,89722,52434.098380,0.936,81.8311,0.038144
3719,3719,Norway,NOR,2019,0.079494,0.015186,0.384156,0.382676,0.020107,0.023429,0.083660,0.010869,0.000423,33122,66481.781690,0.961,82.9552,0.042385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4619,4619,South Africa,ZAF,2019,0.018722,0.003686,0.234371,0.155384,0.077889,0.035025,0.050545,0.016513,0.407865,352693,13366.474640,0.736,66.1750,0.492854
1619,1619,Eswatini,SWZ,2019,0.007482,0.002313,0.206639,0.133179,0.092776,0.036594,0.043395,0.025847,0.451775,7351,7744.536132,0.615,60.5492,0.550460
1529,1529,Equatorial Guinea,GNQ,2019,0.014753,0.003631,0.217885,0.136632,0.050613,0.031548,0.035633,0.028597,0.480708,4406,13447.135720,0.605,61.6444,0.565535
3359,3359,Mozambique,MOZ,2019,0.012034,0.002684,0.249425,0.106493,0.037085,0.020817,0.029584,0.020046,0.521832,127060,1257.939477,0.456,61.1662,0.606832


We remove the five nations used to construct the ratio, since they have an inherent advantage in being close to the independent death ratio.

In [11]:
merged_df = pd.concat([merged_df.sort_values(by='LifeExpectancy',ascending=False).iloc[[0]], merged_df.sort_values(by='LifeExpectancy',ascending=False).iloc[6:]])

In [12]:
merged_df.sort_values(by='Distance')

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,Distance
269,269,Australia,AUS,2019,0.079215,0.016935,0.363760,0.378454,0.029875,0.036697,0.078954,0.015610,0.000500,141893,47312.192000,0.941,83.1100,0.033355
2849,2849,Luxembourg,LUX,2019,0.070365,0.016642,0.388905,0.378102,0.019854,0.034745,0.064526,0.026277,0.000584,3425,76018.564370,0.927,82.1434,0.033901
509,509,Belgium,BEL,2019,0.073728,0.014835,0.374791,0.377745,0.019293,0.033047,0.082477,0.023461,0.000624,89722,52434.098380,0.936,81.8311,0.038144
3719,3719,Norway,NOR,2019,0.079494,0.015186,0.384156,0.382676,0.020107,0.023429,0.083660,0.010869,0.000423,33122,66481.781690,0.961,82.9552,0.042385
1739,1739,France,FRA,2019,0.096542,0.016781,0.349371,0.414037,0.026872,0.026649,0.046290,0.022499,0.000959,476559,46963.195760,0.905,82.7315,0.043308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4619,4619,South Africa,ZAF,2019,0.018722,0.003686,0.234371,0.155384,0.077889,0.035025,0.050545,0.016513,0.407865,352693,13366.474640,0.736,66.1750,0.492854
1619,1619,Eswatini,SWZ,2019,0.007482,0.002313,0.206639,0.133179,0.092776,0.036594,0.043395,0.025847,0.451775,7351,7744.536132,0.615,60.5492,0.550460
1529,1529,Equatorial Guinea,GNQ,2019,0.014753,0.003631,0.217885,0.136632,0.050613,0.031548,0.035633,0.028597,0.480708,4406,13447.135720,0.605,61.6444,0.565535
3359,3359,Mozambique,MOZ,2019,0.012034,0.002684,0.249425,0.106493,0.037085,0.020817,0.029584,0.020046,0.521832,127060,1257.939477,0.456,61.1662,0.606832


We now have the Euclidean distance of each nation's ratio from the independent death ratio.

## Step 3: Compare the similarity between our index and both HDI and Life Expectancy Index

We use Spearman correlation to compare the similarity. First, we drop nations with NaN values in any of the three indexes, since these interfere with the Spearman correlation calculation.

In [22]:
merged_df.dropna(subset=['HDI','LifeExpectancy','Distance'],inplace=True)

merged_df.sort_values(by='Unnamed: 0',inplace=True)

In [23]:
spearmanr(merged_df['Distance'],merged_df['HDI'])

SignificanceResult(statistic=np.float64(-0.5677871743971216), pvalue=np.float64(2.530552417437762e-16))

In [24]:
spearmanr(merged_df['Distance'],merged_df['LifeExpectancy'])

SignificanceResult(statistic=np.float64(-0.6479937304075235), pvalue=np.float64(3.2405551926464884e-22))

Since a lower Distance is better while a higher HDI and Life Expectancy are better, the negative correlations produced by Spearman's test are better interpreted as positive correlations. This suggests that our index, built from the independent death ratio, is comparable to other well-known indicators of a nation's success in addressing chronic disease, such as HDI and Life Expectancy.

# Conclusions

## Independent death ratio (IDR)

We determined an independent death ratio that correlates closely with HDI and Life Expectancy.

In [16]:
# the ratio itself

ratio_df

,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS
2459,0.095564,0.0148,0.371189,0.378563,0.02626,0.036479,0.052411,0.023838,0.000896


## Indexing countries with IDR

We successfully indexed nations by their Euclidean distance to the IDR, finding a moderate correlation with HDI and Life Expectancy.

In [17]:
merged_df.sort_values(by='Distance')

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,Distance
269,269,Australia,AUS,2019,0.079215,0.016935,0.363760,0.378454,0.029875,0.036697,0.078954,0.015610,0.000500,141893,47312.192000,0.941,83.1100,0.033355
2849,2849,Luxembourg,LUX,2019,0.070365,0.016642,0.388905,0.378102,0.019854,0.034745,0.064526,0.026277,0.000584,3425,76018.564370,0.927,82.1434,0.033901
509,509,Belgium,BEL,2019,0.073728,0.014835,0.374791,0.377745,0.019293,0.033047,0.082477,0.023461,0.000624,89722,52434.098380,0.936,81.8311,0.038144
3719,3719,Norway,NOR,2019,0.079494,0.015186,0.384156,0.382676,0.020107,0.023429,0.083660,0.010869,0.000423,33122,66481.781690,0.961,82.9552,0.042385
1739,1739,France,FRA,2019,0.096542,0.016781,0.349371,0.414037,0.026872,0.026649,0.046290,0.022499,0.000959,476559,46963.195760,0.905,82.7315,0.043308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4619,4619,South Africa,ZAF,2019,0.018722,0.003686,0.234371,0.155384,0.077889,0.035025,0.050545,0.016513,0.407865,352693,13366.474640,0.736,66.1750,0.492854
1619,1619,Eswatini,SWZ,2019,0.007482,0.002313,0.206639,0.133179,0.092776,0.036594,0.043395,0.025847,0.451775,7351,7744.536132,0.615,60.5492,0.550460
1529,1529,Equatorial Guinea,GNQ,2019,0.014753,0.003631,0.217885,0.136632,0.050613,0.031548,0.035633,0.028597,0.480708,4406,13447.135720,0.605,61.6444,0.565535
3359,3359,Mozambique,MOZ,2019,0.012034,0.002684,0.249425,0.106493,0.037085,0.020817,0.029584,0.020046,0.521832,127060,1257.939477,0.456,61.1662,0.606832


## Takeaways

The IDR we created, along with its corresponding index, provides a reasonable metric for assessing progress in chronic disease treatment across nations.

A more ambitious application would be to apply this ratio at the county or regional level to identify areas with severely under-addressed chronic health issues. Such an application could help lawmakers accurately demonstrate that specific areas under their jurisdiction need aid for specific chronic diseases.